# ML-KEM seeded from an external bitstream

This notebook runs **ML-KEM (FIPS 203)** with all of its randomness taken from a bitstream file, for example the output of a quantum or chaotic random number generator. It:

1. validates two independent ML-KEM implementations (liboqs and kyber-py) against NIST's official ACVP test vectors;
2. runs key generation, encapsulation and decapsulation with the seeds `d`, `z`, `m` read from your stream, checking that both implementations agree and that implicit rejection works;
3. optionally runs the same procedure with `os.urandom` seeds as a control;
4. writes per-trial and summary CSV files and reports the FIPS 203 entropy requirement.

See **GUIDE.md** for how ML-KEM uses randomness, what each check means, and how to interpret the outputs.

## 1. Install (run once)

liboqs-python compiles the liboqs C library the first time `oqs` is imported, which takes a few minutes. That needs **git, CMake and a C compiler**: on Linux `build-essential cmake git`; on Windows Visual Studio Build Tools ("Desktop development with C++") and Git. After installing, **restart the kernel**.

Note: the PyPI package is `liboqs-python`, not `oqs` (an unrelated package with the same module name).

In [ ]:
%pip install -q -r requirements.txt

## 2. Configuration

Paths are relative to this notebook. Your own results go to `results/run/`; the published results in `results/reference/` are never read or overwritten.

In [ ]:
from pathlib import Path

STREAM_PATH = Path("data/RNG_Bits.txt")   # your bitstream
STREAM_FMT  = "ascii"                     # "ascii": text of 0/1 characters; "bin": raw bytes (MSB first)
START_BIT   = 146_072                     # first stream bit used (bits before it are left for other uses)

PARAM_SETS     = ["ML-KEM-512", "ML-KEM-768", "ML-KEM-1024"]
TRIALS_PER_SET = 500                      # each trial consumes 768 bits (d, z, m)
RUN_CONTROL    = True                     # also run with os.urandom seeds, interleaved

# FIPS 203 entropy check: min-entropy per bit from an SP 800-90B assessment of the stream.
H_MIN                = None               # e.g. 0.741; None skips the check
BITS_ARE_CONDITIONED = False              # True if the stream is full-entropy conditioned output

RESULTS_DIR = Path("results/run")
ACVP_DIR    = Path("acvp")

## 3. Imports and environment

In [ ]:
import csv, hashlib, json, os, platform, statistics, time, urllib.request
from importlib.metadata import version
from operator import xor

import numpy as np
import oqs
from mlkem_backends import LibOQS, KyberPy

ENV = {"python": platform.python_version(), "platform": platform.platform(),
       "liboqs": oqs.oqs_version(), "liboqs-python": version("liboqs-python"),
       "kyber-py": version("kyber-py"), "numpy": np.__version__}
for k, v in ENV.items():
    print(f"{k:14s} {v}")

## 4. Validate both implementations on NIST ACVP vectors

The vectors are downloaded once from usnistgov/ACVP-Server at a pinned commit and checked by SHA-256. Every vector must pass before the stream is used.

In [ ]:
ACVP_COMMIT = "975de31eb83d87039ec88934fdc47d8c312b892d"
ACVP_FILES = {
    "ML-KEM-keyGen-FIPS203":     "d7a62a2c3476957f56dd8d24f9004ea6776ccfe995ffe71a65bb9506dc9c7b1b",
    "ML-KEM-encapDecap-FIPS203": "a556952ce869bb89c3a3196a701dad89647c193a34c86eafb61a9d710d5b810f",
}
ACVP_DIR.mkdir(exist_ok=True)
vectors = {}
for name, sha in ACVP_FILES.items():
    path = ACVP_DIR / f"{name}.json"
    if not path.exists():
        url = (f"https://raw.githubusercontent.com/usnistgov/ACVP-Server/{ACVP_COMMIT}"
               f"/gen-val/json-files/{name}/internalProjection.json")
        urllib.request.urlretrieve(url, path)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == sha, f"{path} does not match the pinned file"
    vectors[name] = json.loads(path.read_text())

def acvp_counts(backend):
    H = bytes.fromhex
    n = {"keyGen": [0, 0], "encaps": [0, 0], "decaps": [0, 0]}
    for g in vectors["ML-KEM-keyGen-FIPS203"]["testGroups"]:
        if g["parameterSet"] == backend.alg:
            for t in g["tests"]:
                ek, dk = backend.keygen(H(t["d"]), H(t["z"]))
                n["keyGen"][0] += (ek == H(t["ek"]) and dk == H(t["dk"])); n["keyGen"][1] += 1
    for g in vectors["ML-KEM-encapDecap-FIPS203"]["testGroups"]:
        if g["parameterSet"] != backend.alg:
            continue
        for t in g["tests"]:
            if g["function"] == "encapsulation":
                K, c = backend.encaps(H(t["ek"]), H(t["m"]))
                n["encaps"][0] += (K == H(t["k"]) and c == H(t["c"])); n["encaps"][1] += 1
            elif g["function"] == "decapsulation":
                n["decaps"][0] += (backend.decaps(H(t["dk"]), H(t["c"])) == H(t["k"])); n["decaps"][1] += 1
    return n

print(f"{'backend':9s} {'parameter set':13s} {'keyGen':>8s} {'encaps':>8s} {'decaps':>8s}")
all_pass = True
for B in (LibOQS, KyberPy):
    for ps in PARAM_SETS:
        n = acvp_counts(B(ps))
        all_pass &= all(ok == total > 0 for ok, total in n.values())
        print(f"{B.name:9s} {ps:13s} " + " ".join(f"{ok:>3d}/{total:<4d}" for ok, total in n.values()))
assert all_pass, "ACVP validation failed; do not use these results"
print("\nAll ACVP vectors pass.")

## 5. Load the stream and plan disjoint segments

Each parameter set gets its own consecutive block of the stream, so no bit is used twice. A fingerprint of the stream and settings is stored in `results/run/run_meta.json`; resuming with a different stream or settings stops with an error instead of mixing results.

In [ ]:
if not STREAM_PATH.exists():
    raise FileNotFoundError(f"{STREAM_PATH} not found: put your bitstream there (see data/README.md)")
raw = STREAM_PATH.read_bytes()
if STREAM_FMT == "ascii":
    bits = np.frombuffer(bytes(b for b in raw if b in b"01"), dtype=np.uint8) - 48
elif STREAM_FMT == "bin":
    bits = np.unpackbits(np.frombuffer(raw, dtype=np.uint8))
else:
    raise ValueError("STREAM_FMT must be 'ascii' or 'bin'")

BITS_PER_TRIAL = 3 * 256
plan, pos = {}, START_BIT
for ps in PARAM_SETS:
    plan[ps] = (pos, pos + TRIALS_PER_SET * BITS_PER_TRIAL)
    pos = plan[ps][1]
if pos > len(bits):
    raise ValueError(f"stream too short: need {pos:,} bits, file has {len(bits):,}")

print(f"{len(bits):,} bits in {STREAM_PATH}")
for ps, (a, b) in plan.items():
    print(f"  {ps:12s} bits [{a:,}, {b:,})")
print(f"  unused after the last segment: {len(bits) - pos:,} bits")

def seed(start):
    # 256 stream bits starting at `start`, packed into 32 bytes, most significant bit first
    return np.packbits(bits[start:start + 256]).tobytes()

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
META = {"stream_sha256": hashlib.sha256(raw).hexdigest(), "stream_format": STREAM_FMT,
        "stream_bits": int(len(bits)), "start_bit": START_BIT, "trials_per_set": TRIALS_PER_SET,
        "param_sets": PARAM_SETS, "run_control": RUN_CONTROL}
meta_path = RESULTS_DIR / "run_meta.json"
if meta_path.exists():
    old = json.loads(meta_path.read_text())
    diff = [k for k in META if old.get(k) != META[k]]
    if diff:
        raise RuntimeError(f"results/run was produced with different {diff}; "
                           "move or delete results/run to start a new run")
meta_path.write_text(json.dumps({**META, "environment": ENV}, indent=2))
print("stream sha256:", META["stream_sha256"])

## 6. FIPS 203 entropy requirement

FIPS 203 (Sec. 3.3) requires the random bit generator to have at least 128 / 192 / 256 bits of security strength for ML-KEM-512 / 768 / 1024. For a raw stream with min-entropy `H_MIN` per bit, a 256-bit seed carries `256 * H_MIN` bits of min-entropy; conditioned full-entropy output counts as 256. See GUIDE.md for the reasoning and its limits.

In [ ]:
REQUIRED = {"ML-KEM-512": 128, "ML-KEM-768": 192, "ML-KEM-1024": 256}
if H_MIN is None:
    print("H_MIN not set: skipping the entropy check")
else:
    per_seed = 256.0 if BITS_ARE_CONDITIONED else 256.0 * H_MIN
    print(f"min-entropy per 256-bit seed: {per_seed:.1f} bits")
    for ps in PARAM_SETS:
        verdict = "meets" if per_seed >= REQUIRED[ps] else "does NOT meet"
        print(f"  {ps:12s} requires {REQUIRED[ps]:3d}: {verdict}")

## 7. Run the trials

Per trial: key generation, encapsulation and decapsulation with liboqs (timed); the same seeds through kyber-py with a byte-for-byte comparison; and an implicit-rejection test with one ciphertext bit flipped. Each row is written to `results/run/mlkem_trials.csv` immediately, so an interrupted run resumes where it stopped.

Timings are Python-level call latencies (`time.perf_counter_ns`), including wrapper overhead; they are not raw C cycle counts.

In [ ]:
TRIALS_CSV = RESULTS_DIR / "mlkem_trials.csv"
FIELDS = ["param_set", "source", "trial", "start_bit", "end_bit",
          "keygen_us", "encaps_us", "decaps_us",
          "K_agree", "xcheck_ek", "xcheck_dk", "xcheck_c", "xcheck_K", "implicit_reject_ok",
          "len_ek", "len_dk", "len_c", "K_hex"]

done = set()
if TRIALS_CSV.exists():
    with TRIALS_CSV.open() as fh:
        done = {(r["param_set"], r["source"], int(r["trial"])) for r in csv.DictReader(fh)}
sources = ["qrng", "urandom"] if RUN_CONTROL else ["qrng"]
jobs = [(ps, src, i) for ps in PARAM_SETS for i in range(TRIALS_PER_SET) for src in sources]
jobs = [j for j in jobs if j not in done]
print(f"{len(done)} trials already done, {len(jobs)} to run")

primary = {ps: LibOQS(ps) for ps in PARAM_SETS}
reference = {ps: KyberPy(ps) for ps in PARAM_SETS}

def elapsed_us(t0):
    return (time.perf_counter_ns() - t0) / 1000.0

new_file = not TRIALS_CSV.exists()
t_start = time.time()
with TRIALS_CSV.open("a", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=FIELDS)
    if new_file:
        writer.writeheader()
    for n, (ps, src, i) in enumerate(jobs, 1):
        if src == "qrng":
            s = plan[ps][0] + i * BITS_PER_TRIAL
            d, z, m = seed(s), seed(s + 256), seed(s + 512)
            start_bit, end_bit = s, s + BITS_PER_TRIAL
        else:
            d, z, m = os.urandom(32), os.urandom(32), os.urandom(32)
            start_bit = end_bit = ""
        P, R = primary[ps], reference[ps]

        t0 = time.perf_counter_ns(); ek, dk = P.keygen(d, z); t_keygen = elapsed_us(t0)
        t0 = time.perf_counter_ns(); K, c = P.encaps(ek, m);  t_encaps = elapsed_us(t0)
        t0 = time.perf_counter_ns(); K_dec = P.decaps(dk, c); t_decaps = elapsed_us(t0)

        ek_r, dk_r = R.keygen(d, z)
        K_r, c_r = R.encaps(ek_r, m)

        c_bad = bytes([xor(c[0], 1)]) + c[1:]            # flip the least-significant bit of byte 0
        K_bad, K_bad_r = P.decaps(dk, c_bad), R.decaps(dk_r, c_bad)

        writer.writerow({
            "param_set": ps, "source": src, "trial": i, "start_bit": start_bit, "end_bit": end_bit,
            "keygen_us": round(t_keygen, 2), "encaps_us": round(t_encaps, 2), "decaps_us": round(t_decaps, 2),
            "K_agree": int(K == K_dec),
            "xcheck_ek": int(ek == ek_r), "xcheck_dk": int(dk == dk_r),
            "xcheck_c": int(c == c_r), "xcheck_K": int(K == K_r),
            "implicit_reject_ok": int(K_bad != K and K_bad == K_bad_r),
            "len_ek": len(ek), "len_dk": len(dk), "len_c": len(c), "K_hex": K.hex()})
        fh.flush(); os.fsync(fh.fileno())
        if n % 250 == 0 or n == len(jobs):
            print(f"  {n}/{len(jobs)} trials, {time.time() - t_start:.0f} s")

## 8. Summary

Written to `results/run/mlkem_summary.csv`. Every check should read `n/n`.

In [ ]:
SUMMARY_CSV = RESULTS_DIR / "mlkem_summary.csv"
CHECKS = ["K_agree", "xcheck_ek", "xcheck_dk", "xcheck_c", "xcheck_K", "implicit_reject_ok"]
with TRIALS_CSV.open() as fh:
    rows = list(csv.DictReader(fh))

summary = []
for ps in PARAM_SETS:
    for src in ["qrng", "urandom"]:
        R = [r for r in rows if r["param_set"] == ps and r["source"] == src]
        if not R:
            continue
        s = {"param_set": ps, "source": src, "n": len(R),
             **{c: f"{sum(int(r[c]) for r in R)}/{len(R)}" for c in CHECKS},
             "len_ek": R[0]["len_ek"], "len_dk": R[0]["len_dk"], "len_c": R[0]["len_c"],
             "unique_K": len({r["K_hex"] for r in R})}
        for op in ["keygen_us", "encaps_us", "decaps_us"]:
            v = sorted(float(r[op]) for r in R)
            s[op + "_median"] = round(statistics.median(v), 1)
            s[op + "_p95"] = round(v[int(0.95 * (len(v) - 1))], 1)
        summary.append(s)

with SUMMARY_CSV.open("w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(summary[0])); w.writeheader(); w.writerows(summary)

print(f"{'parameter set':13s} {'source':8s} {'n':>4s} {'K agree':>8s} {'x-check':>8s} {'rejection':>9s} "
      f"{'|ek|':>5s} {'|c|':>5s} {'keygen':>7s} {'encaps':>7s} {'decaps':>7s}")
for s in summary:
    xc = min((s[c] for c in ["xcheck_ek", "xcheck_dk", "xcheck_c", "xcheck_K"]), key=lambda v: int(v.split("/")[0]))
    print(f"{s['param_set']:13s} {s['source']:8s} {s['n']:>4d} {s['K_agree']:>8s} {xc:>8s} "
          f"{s['implicit_reject_ok']:>9s} {s['len_ek']:>5s} {s['len_c']:>5s} "
          f"{s['keygen_us_median']:>7.1f} {s['encaps_us_median']:>7.1f} {s['decaps_us_median']:>7.1f}")
print("(timings: median microseconds)")
ok = all(s[c].split("/")[0] == s[c].split("/")[1] for s in summary for c in CHECKS)
print("\nAll checks pass." if ok else "\nSome checks FAILED: inspect mlkem_trials.csv")

## 9. Optional: reproduce the published results

If your stream is the one used for `results/reference/` (same SHA-256 as in `results/reference/run_meta.json`) and the settings are the defaults, every QRNG shared key must match the reference exactly. Control (`os.urandom`) keys and timings are expected to differ.

In [ ]:
REF = Path("results/reference")
ref_meta_path = REF / "run_meta.json"
if not ref_meta_path.exists():
    print("no reference results found")
else:
    ref_meta = json.loads(ref_meta_path.read_text())
    if ref_meta.get("stream_sha256") != META["stream_sha256"]:
        print("your stream differs from the reference stream: nothing to compare")
    else:
        with (REF / "mlkem_trials.csv").open() as fh:
            ref_K = {(r["param_set"], r["trial"]): r["K_hex"] for r in csv.DictReader(fh) if r["source"] == "qrng"}
        mine = {(r["param_set"], r["trial"]): r["K_hex"] for r in rows if r["source"] == "qrng"}
        common = set(ref_K) & set(mine)
        same = sum(ref_K[k] == mine[k] for k in common)
        print(f"QRNG shared keys identical to the reference: {same}/{len(common)}")